In [1]:
!pip install sentence-transformers numpy pandas

   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 4.3 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.6 MB 5.6 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.6 MB 5.9 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/12.6 MB 5.9 MB/s eta 0:00:02
   -------------- ------------------------- 4.7/12.6 MB 6.0 MB/s eta 0:00:02
   ------------------- -------------------- 6.0/12.6 MB 5.9 MB/s eta 0:00:02
   ----------------------- ---------------- 7.3/12.6 MB 5.9 MB/s eta 0:00:01
   -------------------------- ------------- 8.4/12.6 MB 6.0 MB/s eta 0:00:01
   ----------------------------- ---------- 9.4/12.6 MB 5.9 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.6 MB 5.8 MB/s eta 0:00:01
   ----------------------------------- ---- 11.3/12.6 MB 5.7 MB/s eta 0:00:01
   -------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [4]:
#Setting project paths
project_dir = os.path.join(
    os.path.expanduser("~"),
    "AI_Software_Engineering_Agent"
)

processed_dir = os.path.join(
    project_dir,
    "data",
    "processed_code"
)

os.makedirs(processed_dir, exist_ok=True)

print("Project directory:")
print(project_dir)

print("\nProcessed directory:")
print(processed_dir)

Project directory:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent

Processed directory:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code


In [5]:
#Loading the semantic chunks
chunks_path = os.path.join(
    processed_dir,
    "code_chunks.json"
)

with open(chunks_path, "r", encoding="utf-8") as f:
    code_chunks = json.load(f)

print("Total code chunks:", len(code_chunks))

Total code chunks: 30650


In [6]:
print(code_chunks[0].keys())

dict_keys(['chunk_type', 'name', 'parent_class', 'line_start', 'line_end', 'arguments', 'docstring', 'code', 'searchable_text', 'chunk_id', 'file_id', 'repository', 'repository_url', 'file_name', 'file_path', 'language'])


In [7]:
print("Repository:", code_chunks[0]["repository"])
print("File:", code_chunks[0]["file_name"])
print("Type:", code_chunks[0]["chunk_type"])
print("Name:", code_chunks[0]["name"])
print("\nSearchable text:\n")
print(code_chunks[0]["searchable_text"][:2000])

Repository: requests
File: flask_theme_support.py
Type: class
Name: FlaskyStyle

Searchable text:

Repository: requests
File: flask_theme_support.py
Path: docs\_themes\flask_theme_support.py
Type: class
Class: FlaskyStyle
Docstring: 
Code:
class FlaskyStyle(Style):
    background_color = "#f8f8f8"
    default_style = ""

    styles = {
        # No corresponding class for the following:
        #Text:                     "", # class:  ''
        Whitespace:                "underline #f8f8f8",      # class: 'w'
        Error:                     "#a40000 border:#ef2929", # class: 'err'
        Other:                     "#000000",                # class 'x'

        Comment:                   "italic #8f5902", # class: 'c'
        Comment.Preproc:           "noitalic",       # class: 'cp'

        Keyword:                   "bold #004461",   # class: 'k'
        Keyword.Constant:          "bold #004461",   # class: 'kc'
        Keyword.Declaration:       "bold #004461",   # class: 'kd'


In [8]:
# Loading the embedding model
model_name = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Tushti Gupta\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tushti Gupta\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [9]:
#Preparing the text
texts = [
    chunk["searchable_text"]
    for chunk in code_chunks
]

print("Number of texts:", len(texts))

Number of texts: 30650


In [10]:
# Generate embeddings
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding generation completed.")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/958 [00:00<?, ?it/s]

Embedding generation completed.
Embedding shape: (30650, 384)


In [11]:
embeddings.shape

(30650, 384)

In [12]:
#Checking the embeddings
print("Data type:", embeddings.dtype)
print("Shape:", embeddings.shape)
print("Minimum value:", embeddings.min())
print("Maximum value:", embeddings.max())

Data type: float32
Shape: (30650, 384)
Minimum value: -0.2793855
Maximum value: 0.26948375


In [13]:
print(embeddings[0])

[-6.15325347e-02  3.60520445e-02 -1.89522039e-02  7.32468590e-02
  9.57318172e-02 -5.26010618e-02  2.27881502e-02 -1.80565205e-03
 -3.25722918e-02 -6.49579465e-02 -5.23350202e-02  6.04945123e-02
  2.77111083e-02 -3.52643686e-03  5.85779510e-02  3.77247296e-02
 -6.91974983e-02 -2.78352071e-02 -5.93433566e-02  4.55886647e-02
  7.13831112e-02  7.48768970e-02  1.41267195e-01  8.06550607e-02
 -4.68932800e-02 -5.21274768e-02 -4.85971458e-02  4.27337438e-02
 -2.69835927e-02 -1.23544205e-02  3.24246637e-03  4.89346683e-02
  7.23938197e-02  2.97388937e-02 -1.00199813e-02  1.82972532e-02
  1.86153110e-02 -2.54947878e-02  2.10422203e-02  9.27379578e-02
 -4.69527692e-02  1.15400609e-02 -1.51821487e-02 -2.38338131e-02
 -5.66065498e-02 -4.67521958e-02 -2.43189000e-02  1.73239559e-02
 -2.47497596e-02 -1.75951179e-02 -4.21354547e-02  3.21177132e-02
 -4.76433337e-02 -8.08295384e-02 -5.01547270e-02 -2.99420301e-03
  1.32160848e-02  3.28339152e-02  3.20499763e-02 -2.10454985e-02
  5.91202714e-02 -2.34840

In [14]:
#Save the embeddings
embeddings_path = os.path.join(
    processed_dir,
    "embeddings.npy"
)

np.save(
    embeddings_path,
    embeddings
)

print("Embeddings saved successfully.")
print(embeddings_path)

Embeddings saved successfully.
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\processed_code\embeddings.npy


In [15]:
#Verifying the embedding file
loaded_embeddings = np.load(embeddings_path)

print("Loaded embedding shape:")
print(loaded_embeddings.shape)

Loaded embedding shape:
(30650, 384)


In [17]:
#Creating embedding meta data file
embedding_metadata = []

for i, chunk in enumerate(code_chunks):

    embedding_metadata.append({
        "embedding_id": i,
        "chunk_id": chunk["chunk_id"],
        "repository": chunk["repository"],
        "file_name": chunk["file_name"],
        "file_path": chunk["file_path"],
        "chunk_type": chunk["chunk_type"],
        "name": chunk["name"],
        "parent_class": chunk.get("parent_class")
    })

In [18]:
metadata_path = os.path.join(
    processed_dir,
    "embedding_metadata.json"
)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        embedding_metadata,
        f,
        indent=2
    )

print("Embedding metadata saved.")

Embedding metadata saved.


In [19]:
#Test semantic simalarity
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
query_embedding = embeddings[0].reshape(1, -1)

In [21]:
similarities = cosine_similarity(
    query_embedding,
    embeddings
)[0]

In [22]:
top_indices = np.argsort(similarities)[-5:][::-1]

In [23]:
for rank, idx in enumerate(top_indices):

    chunk = code_chunks[idx]

    print("=" * 70)
    print("Rank:", rank + 1)
    print("Similarity:", round(similarities[idx], 4))
    print("Repository:", chunk["repository"])
    print("File:", chunk["file_name"])
    print("Type:", chunk["chunk_type"])
    print("Name:", chunk["name"])
    print("=" * 70)

Rank: 1
Similarity: 1.0
Repository: requests
File: flask_theme_support.py
Type: class
Name: FlaskyStyle
Rank: 2
Similarity: 0.6545
Repository: flask
File: helpers.py
Type: method
Name: __init__
Rank: 3
Similarity: 0.6476
Repository: flask
File: cli.py
Type: function
Name: get_version
Rank: 4
Similarity: 0.6361
Repository: flask
File: blueprints.py
Type: class
Name: Blueprint
Rank: 5
Similarity: 0.6301
Repository: flask
File: test_async.py
Type: class
Name: AppError
